# ARM — Lựa chọn cấu hình và đánh giá Test

27 cấu hình được so sánh trên Train/Validation với tám nhóm horizon × hướng. Danh sách luật và thứ tự theo Validation được cố định trước khi đánh giá trên Test.

## A. Cấu hình và hàm đọc dữ liệu

In [1]:
from pathlib import Path
from itertools import combinations, product
from datetime import datetime, timezone
import argparse
import hashlib
import json
import math

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/modeling/horizons').is_dir() and (p/'notebook').is_dir())
OUT = ROOT / 'results/arm'
DATA_OUT = ROOT / 'data/arm'
HORIZONS = (1, 5, 10, 20)
FEATURES = [
    'Gold_Daily_Return', 'Gold_Return_Lag_1', 'Gold_Intraday_Range',
    'Gold_Volatility_20', 'Gold_Distance_to_MA_20', 'Gold_RSI_14',
    'Gold_MACD_Histogram', 'Oil_Price_Change', 'Dollar_Daily_Return',
    'SP500_Daily_Return', 'SP500_Drawdown',
]
QUANTILE_FEATURES = ['Gold_Intraday_Range', 'Gold_Volatility_20',
                     'Gold_RSI_14', 'SP500_Drawdown']
GRID = list(product((0.02, 0.05, 0.10), (0.50, 0.55, 0.60), (1.00, 1.05, 1.10)))
TARGET = 'Gold_Target_Next'
MIN_VAL_COUNT = 30


from IPython.display import display
import pickle
for stage in ['1_train','2_validation','3_test']:
    (OUT/stage).mkdir(parents=True,exist_ok=True)


In [2]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_data(h):
    source = ROOT / f'data/modeling/horizons/h{h}'
    data = pd.read_csv(source / 'modeling_dataset.csv', parse_dates=['Date'])
    trace = pd.read_csv(source / 'target_trace.csv', parse_dates=['Date', 'Target_Date'])
    data = data.merge(trace[['Date', 'Target_Date', 'Close', 'Target_Close']],
                      on='Date', validate='one_to_one')
    if data[FEATURES + [TARGET]].isna().any().any() or data.Date.duplicated().any():
        raise ValueError('Missing feature/target or duplicate date')
    if not data.Date.is_monotonic_increasing or not data.Target_Date.gt(data.Date).all():
        raise ValueError('Invalid date order')
    if not data[TARGET].eq((data.Target_Close > data.Close).astype(int)).all():
        raise ValueError('Target disagrees with price trace')
    return data

def split_masks(data):
    
    return {
        'train': (data.Date < '2021-01-01') & (data.Target_Date < '2021-01-01'),
        'val': data.Date.between('2021-01-01', '2023-12-31') & (data.Target_Date < '2024-01-01'),
        'test': data.Date >= '2024-01-01',
    }

In [3]:
def fit_thresholds(train):
    rows = []
    for name in QUANTILE_FEATURES:
        q1, q2 = train[name].quantile([1/3, 2/3], interpolation='linear')
        rows.append({'Feature': name, 'q1': float(q1), 'q2': float(q2),
                     'Train_Rows': len(train), 'Interpolation': 'linear'})
    return pd.DataFrame(rows)

def encode(frame, thresholds, h):
    """Encode original unscaled values; quantile edges are fitted on Train only."""
    encoded = frame[['Date', 'Target_Date']].copy()
    cuts = thresholds.set_index('Feature')
    for name in FEATURES:
        values = frame[name]
        if name in QUANTILE_FEATURES:
            states = ['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown' else ['Low', 'Medium', 'High']
            q1, q2 = cuts.loc[name, ['q1', 'q2']]
            state = np.select([values <= q1, values <= q2], states[:2], default=states[2])
        else:
            
            state = np.select([values < 0, values == 0], ['Negative', 'Zero'], default='Positive')
        encoded[name] = name + '=' + pd.Series(state, index=frame.index)
    encoded['Target_Item'] = np.where(frame[TARGET] == 1, f'Gold_Up_H{h}', f'Gold_NonUp_H{h}')
    return encoded

def item_names(h):
    names = []
    for name in FEATURES:
        states = (['Deep', 'Medium', 'Shallow'] if name == 'SP500_Drawdown'
                  else ['Low', 'Medium', 'High'] if name in QUANTILE_FEATURES
                  else ['Negative', 'Zero', 'Positive'])
        names.extend(f'{name}={state}' for state in states)
    return names + [f'Gold_NonUp_H{h}', f'Gold_Up_H{h}']

def onehot(encoded, h):
    matrix = {}
    for item in item_names(h):
        field = item.split('=')[0] if '=' in item else 'Target_Item'
        matrix[item] = encoded[field].eq(item).to_numpy()
    return pd.DataFrame(matrix, index=encoded.index)

In [4]:
def measure_rules(rules, matrix, prefix):
    """Support is P(antecedent AND consequent), not antecedent coverage."""
    rows = []
    for rule in rules.itertuples(index=False):
        antecedent = matrix[rule.Antecedent.split(' & ')].all(axis=1).to_numpy()
        consequent = matrix[rule.Consequent].to_numpy()
        n_a = int(antecedent.sum())
        n_joint = int((antecedent & consequent).sum())
        base = float(consequent.mean())
        conf = n_joint / n_a if n_a else np.nan
        rows.append({f'{prefix}_Rows': len(matrix), f'{prefix}_Antecedent_Count': n_a,
                     f'{prefix}_Joint_Count': n_joint, f'{prefix}_Coverage': n_a / len(matrix),
                     f'{prefix}_Support': n_joint / len(matrix), f'{prefix}_Base_Rate': base,
                     f'{prefix}_Confidence': conf, f'{prefix}_Lift': conf / base if base else np.nan,
                     f'{prefix}_Leverage': n_joint / len(matrix) - n_a / len(matrix) * base})
    return pd.concat([rules.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

In [5]:
def grid_filter(rules, config):
    support, confidence, lift = config
    return (rules.Train_Support >= support) & (rules.Train_Confidence >= confidence) & (rules.Train_Lift >= lift)

## B. Lựa chọn cấu hình trên Validation

Điểm lựa chọn là trung bình median lift Val của tám nhóm. Mỗi nhóm có ít nhất một luật với antecedent xuất hiện ≥30 lần và lift Val >1. Cấu hình không đáp ứng đủ tám nhóm không đủ điều kiện đánh giá Test.

In [6]:
def choose_configuration():
    """One shared grid choice, scored equally across 4 horizons x 2 directions."""
    tables = {h: pd.read_csv(OUT / f'1_train/H{h}_rules.csv') for h in HORIZONS}
    scores, details = [], []
    for config in GRID:
        support, confidence, lift = config
        group_medians, group_lengths = [], []
        for h, rules in tables.items():
            passing = rules.loc[grid_filter(rules, config)]
            stable = passing.loc[passing.Val_Stable]
            for direction in ['Up', 'NonUp']:
                group = stable.loc[stable.Direction == direction]
                group_medians.append(float(group.Val_Lift.median()) if len(group) else np.nan)
                group_lengths.append(float(group.Length.mean()) if len(group) else np.nan)
                details.append({'Min_Support': support, 'Min_Confidence': confidence, 'Min_Lift': lift,
                                'Horizon': h, 'Direction': direction, 'Stable_Rules': len(group),
                                'External_Stable_Rules': int(group.External.sum()),
                                'Median_Val_Lift': group_medians[-1]})
        eligible = not any(np.isnan(group_medians))
        scores.append({'Min_Support': support, 'Min_Confidence': confidence, 'Min_Lift': lift,
                       'Eligible': eligible, 'Groups_Covered': int(np.isfinite(group_medians).sum()),
                       'Balanced_Median_Val_Lift': float(np.mean(group_medians)) if eligible else np.nan,
                       'Balanced_Mean_Length': float(np.mean(group_lengths)) if eligible else np.nan})
    scores = pd.DataFrame(scores)
    print('So sánh 27 cấu hình trên Validation:'); display(scores)
    print('Chi tiết 8 nhóm:'); display(pd.DataFrame(details))
    eligible = scores.loc[scores.Eligible].sort_values(
        ['Balanced_Median_Val_Lift', 'Balanced_Mean_Length', 'Min_Support', 'Min_Confidence', 'Min_Lift'],
        ascending=[False, True, False, False, False], kind='stable')
    if eligible.empty:
        print('No eligible configuration. Test remains unevaluated.', flush=True)
        return False
    best = eligible.iloc[0]
    config = tuple(float(best[x]) for x in ['Min_Support', 'Min_Confidence', 'Min_Lift'])
    locked = {'status': 'LOCKED_BEFORE_TEST', 'locked_at_utc': datetime.now(timezone.utc).isoformat(),
              'min_support': config[0], 'min_confidence': config[1], 'min_lift': config[2],
              'score': float(best.Balanced_Median_Val_Lift),
              'selection': 'Mean of eight within-group median Val lifts; all eight groups required',
              'rule_pool': 'Gold-only and external rules; external rules prioritized in top reports',
              'stability': 'Val antecedent count >= 30 AND Val lift > 1',
              'tie_break': 'Smaller balanced mean length; then higher support/confidence/lift',
              'horizons': {}}
    for h, rules in tables.items():
        selected = rules.loc[grid_filter(rules, config) & rules.Val_Stable].copy()
        selected = selected.sort_values(['Direction', 'External', 'Val_Lift', 'Val_Joint_Count', 'Length', 'Rule_ID'],
                                        ascending=[True, False, False, False, True, True], kind='stable')
        selected['Val_Report_Rank'] = selected.groupby('Direction').cumcount() + 1
        selected['Top10_Report'] = selected.Val_Report_Rank <= 10
        for key, value in zip(['min_support','min_confidence','min_lift'],config):
            selected[key] = value
        destination = OUT/f'2_validation/H{h}_selected_rules.csv'
        selected.to_csv(destination,index=False)
        locked['horizons'][str(h)] = {
            'rules_sha256': digest(destination),
            'candidate_sha256': digest(OUT/f'1_train/H{h}_rules.csv'),
            'encoding_sha256': digest(DATA_OUT/f'ARM-H{h}/train_encoding.pkl')}
    
    global LOCKED_SELECTION
    LOCKED_SELECTION = locked
    print('Locked shared configuration:', config, 'score=', locked['score'], flush=True)
    return True

In [7]:
if not choose_configuration():
    raise RuntimeError('Không có cấu hình đạt đủ 8 nhóm; chưa đánh giá Test.')

So sánh 27 cấu hình trên Validation:


,Min_Support,Min_Confidence,Min_Lift,Eligible,Groups_Covered,Balanced_Median_Val_Lift,Balanced_Mean_Length
0,0.02,0.50,1.00,True,8,1.087585,2.871247
1,0.02,0.50,1.05,True,8,1.086985,2.917048
2,0.02,0.50,1.10,True,8,1.084356,2.952629
3,0.02,0.55,1.00,True,8,1.078252,2.906105
4,0.02,0.55,1.05,True,8,1.077620,2.935408
5,0.02,0.55,1.10,True,8,1.073194,2.963163
6,0.02,0.60,1.00,False,5,NaN,NaN
7,0.02,0.60,1.05,False,5,NaN,NaN
8,0.02,0.60,1.10,False,5,NaN,NaN
9,0.05,0.50,1.00,True,8,1.073867,2.567727


Chi tiết 8 nhóm:


,Min_Support,Min_Confidence,Min_Lift,Horizon,Direction,Stable_Rules,External_Stable_Rules,Median_Val_Lift
0,0.02,0.5,1.0,1,Up,548,429,1.072019
1,0.02,0.5,1.0,1,NonUp,287,220,1.101235
2,0.02,0.5,1.0,5,Up,428,331,1.056731
3,0.02,0.5,1.0,5,NonUp,48,35,1.062584
4,0.02,0.5,1.0,10,Up,400,288,1.075361
...,...,...,...,...,...,...,...,...
211,0.10,0.6,1.1,5,NonUp,0,0,NaN
212,0.10,0.6,1.1,10,Up,0,0,NaN
213,0.10,0.6,1.1,10,NonUp,0,0,NaN
214,0.10,0.6,1.1,20,Up,6,2,1.093326


Locked shared configuration: (0.02, 0.5, 1.0) score= 1.08758499249041


## C. Khóa luật và đánh giá Test
Từ đây mới tính kết quả Test, không dùng Test để chọn lại cấu hình hoặc thay đổi danh sách luật.

In [8]:
def check_lock():
    
    for h in HORIZONS:
        meta = LOCKED_SELECTION['horizons'][str(h)]
        paths = {
            'rules_sha256': OUT/f'2_validation/H{h}_selected_rules.csv',
            'candidate_sha256': OUT/f'1_train/H{h}_rules.csv',
            'encoding_sha256': DATA_OUT/f'ARM-H{h}/train_encoding.pkl'}
        for key,path in paths.items():
            if digest(path) != meta[key]:
                raise ValueError(f'H{h}: input changed after selection; rerun Train/Val first')
        state = pickle.loads(paths['encoding_sha256'].read_bytes())
        for name,key in [('modeling_dataset.csv','source_sha256'),('target_trace.csv','trace_sha256')]:
            if digest(ROOT/f'data/modeling/horizons/h{h}'/name) != state['preparation'][key]:
                raise ValueError('Source changed since Train encoding')
    return LOCKED_SELECTION


In [9]:
def evaluate_test():
    check_lock()
    summaries = []
    for h in HORIZONS:
        selected = pd.read_csv(OUT/f'2_validation/H{h}_selected_rules.csv')
        state = pickle.loads((DATA_OUT/f'ARM-H{h}/train_encoding.pkl').read_bytes())
        data = load_data(h)
        masks = split_masks(data)
        encoded = encode(data.loc[masks['test']],state['thresholds'],h)
        matrix = onehot(encoded,h)
        encoded.to_csv(DATA_OUT/f'ARM-H{h}/ARM_test.csv',index=False)
        pd.concat([encoded[['Date','Target_Date']],matrix.astype('uint8')],axis=1).to_csv(
            DATA_OUT/f'ARM-H{h}/ARM_test_onehot.csv',index=False)
        evaluated = measure_rules(selected,matrix,'Test')
        evaluated['Test_Lift_Above_1'] = evaluated.Test_Lift.gt(1)
        evaluated['Test_Enough_Occurrences'] = evaluated.Test_Antecedent_Count.ge(30)
        evaluated['Test_Assessment'] = np.select(
            [evaluated.Test_Lift.isna(),
             evaluated.Test_Lift_Above_1 & evaluated.Test_Enough_Occurrences,
             evaluated.Test_Lift_Above_1],
            ['Insufficient data','Lift > 1; at least 30 occurrences',
             'Lift > 1; fewer than 30 occurrences'], default='Lift <= 1')
        evaluated.to_csv(OUT/f'3_test/H{h}_evaluated_rules.csv',index=False)
        candidates = pd.read_csv(OUT/f'1_train/H{h}_rules.csv')
        summaries.append({
            'Horizon':h, 'Train_Candidate_Rules':len(candidates),
            'Validation_Selected_Rules':len(selected), 'Test_Evaluated_Rules':len(evaluated),
            'Test_Lift_Above_1_At_Least_30':int((evaluated.Test_Lift_Above_1 & evaluated.Test_Enough_Occurrences).sum()),
            'Test_Lift_Above_1_Fewer_Than_30':int((evaluated.Test_Lift_Above_1 & ~evaluated.Test_Enough_Occurrences).sum()),
            'Test_Lift_At_Most_1':int(evaluated.Test_Lift.le(1).sum()),
            'Test_Undefined_Lift':int(evaluated.Test_Lift.isna().sum()),
            'Median_Test_Support':evaluated.Test_Support.median(),
            'Median_Test_Confidence':evaluated.Test_Confidence.median(),
            'Median_Test_Lift':evaluated.Test_Lift.median()})
    pd.DataFrame(summaries).to_csv(OUT/'3_test/summary.csv',index=False)
    print('Đã đánh giá toàn bộ luật đã chốt, không xóa luật yếu trên Test.')


In [10]:
evaluate_test()

Đã đánh giá toàn bộ luật đã chốt, không xóa luật yếu trên Test.


## D. Tổng hợp kết quả

In [11]:
print('Số luật qua Train → Validation → Test:'); display(pd.read_csv(OUT/'3_test/summary.csv'))


Số luật qua Train → Validation → Test:


,Horizon,Train_Candidate_Rules,Validation_Selected_Rules,Test_Evaluated_Rules,Test_Lift_Above_1_At_Least_30,Test_Lift_Above_1_Fewer_Than_30,Test_Lift_At_Most_1,Test_Undefined_Lift,Median_Test_Support,Median_Test_Confidence,Median_Test_Lift
0,1,4374,835,835,449,70,311,5,0.054173,0.546589,1.032188
1,5,4341,476,476,261,49,166,0,0.058565,0.638653,1.044986
2,10,4328,442,442,181,55,197,9,0.053441,0.612903,1.011506
3,20,4335,536,536,151,64,316,5,0.046852,0.626667,0.957006
